<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 06 — Gold: Panel municipal año (mesa analítica final)

Cruza las 4 fuentes Silver en una sola **tabla de panel** con grano
`(COD_MPIO, ANO)`. Esta es la mesa que alimenta EDA y modelos ML.

**Plan de joins:**
1. **ICFES → municipal-año**: avg(PUNT_GLOBAL), avg(PUNT_*), n_estudiantes,
   pct_internet, pct_computador, pct_rural, pct_oficial.
2. **Internet → municipal-año**: total_accesos, avg_velocidad_bajada, n_proveedores.
3. **MEN → municipal-año**: cobertura_neta, deserción, aprobación, sedes_conectadas.
4. **SISBEN → municipal (sin año)**: idx_privacion, pct_grupo_A, pct_rural, n_personas.

Join principal: `LEFT JOIN` partiendo de ICFES (que es el outcome de interés).
SISBEN se broadcasta porque es chico (~1100 filas, una por municipio).

## 1. Setup y carga de Silver

In [1]:
import sys
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark = get_spark("Entrega2-Gold-Panel", executor_memory="4g", driver_memory="2g", cores=2)

icfes    = spark.read.parquet(P.SILVER_ICFES)
internet = spark.read.parquet(P.SILVER_INTERNET)
sisben   = spark.read.parquet(P.SILVER_SISBEN_MPIO)
men      = spark.read.parquet(P.SILVER_MEN)

for name, df in [("icfes",icfes),("internet",internet),("sisben",sisben),("men",men)]:
    print(f"{name:>8}: {df.count():>10,} filas   {len(df.columns)} cols")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/22 08:17:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


   icfes:  4,500,067 filas   27 cols


internet:  2,792,934 filas   12 cols


  sisben:      1,099 filas   25 cols


     men:     15,700 filas   41 cols


## 2. Agregar ICFES a (COD_MPIO, ANO)

In [2]:
icfes_agg = (icfes
    .groupBy("COD_MPIO", "ANO", "COLE_DEPTO_UBICACION")
    .agg(
        F.count("*").alias("n_estudiantes"),
        F.round(F.avg("PUNT_GLOBAL"), 2).alias("avg_punt_global"),
        F.round(F.stddev("PUNT_GLOBAL"), 2).alias("sd_punt_global"),
        F.round(F.percentile_approx("PUNT_GLOBAL", 0.5), 1).alias("med_punt_global"),
        F.round(F.avg("PUNT_LECTURA_CRITICA"), 2).alias("avg_punt_lectura"),
        F.round(F.avg("PUNT_MATEMATICAS"), 2).alias("avg_punt_matematicas"),
        F.round(F.avg("PUNT_C_NATURALES"), 2).alias("avg_punt_naturales"),
        F.round(F.avg("PUNT_SOCIALES_CIUDADANAS"), 2).alias("avg_punt_sociales"),
        F.round(F.avg("PUNT_INGLES"), 2).alias("avg_punt_ingles"),
        F.round(F.avg("TIENE_INTERNET_BIN"), 4).alias("pct_internet_icfes"),
        F.round(F.avg("TIENE_COMPUTADOR_BIN"), 4).alias("pct_computador_icfes"),
        F.round(F.avg(F.when(F.col("COLE_AREA_UBICACION")=="RURAL", 1.0).otherwise(0.0)), 4).alias("pct_rural_colegio"),
        F.round(F.avg(F.when(F.col("COLE_NATURALEZA")=="OFICIAL", 1.0).otherwise(0.0)), 4).alias("pct_colegio_oficial"),
    )
)
print(f"ICFES agg rows (mpio-año): {icfes_agg.count():,}")
icfes_agg.orderBy(F.desc("avg_punt_global")).show(5, truncate=False)

26/05/22 08:18:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


ICFES agg rows (mpio-año): 7,004


+--------+----+--------------------+-------------+---------------+--------------+---------------+----------------+--------------------+------------------+-----------------+---------------+------------------+--------------------+-----------------+-------------------+
|COD_MPIO|ANO |COLE_DEPTO_UBICACION|n_estudiantes|avg_punt_global|sd_punt_global|med_punt_global|avg_punt_lectura|avg_punt_matematicas|avg_punt_naturales|avg_punt_sociales|avg_punt_ingles|pct_internet_icfes|pct_computador_icfes|pct_rural_colegio|pct_colegio_oficial|
+--------+----+--------------------+-------------+---------------+--------------+---------------+----------------+--------------------+------------------+-----------------+---------------+------------------+--------------------+-----------------+-------------------+
|68276   |2021|SANTANDER           |119          |369.49         |40.0          |374            |72.16           |78.82               |69.91             |70.29            |87.17          |0.9328     

## 3. Agregar Internet Fijo a (COD_MPIO, ANO)

In [3]:
internet_agg = (internet
    .groupBy("COD_MPIO", "ANO")
    .agg(
        F.sum("NUM_ACCESOS").alias("total_accesos"),
        F.countDistinct("PROVEEDOR").alias("n_proveedores"),
        F.round(F.avg("VELOCIDAD_BAJADA"), 2).alias("avg_velocidad_bajada"),
        F.round(F.avg("VELOCIDAD_SUBIDA"), 2).alias("avg_velocidad_subida"),
        F.countDistinct("TECNOLOGIA").alias("n_tecnologias"),
        F.sum(F.when(F.col("SEGMENTO")=="RESIDENCIAL", F.col("NUM_ACCESOS")).otherwise(0)).alias("accesos_residenciales"),
        F.sum(F.when(F.col("SEGMENTO")=="CORPORATIVO", F.col("NUM_ACCESOS")).otherwise(0)).alias("accesos_corporativos"),
    )
)
print(f"Internet agg rows (mpio-año): {internet_agg.count():,}")
internet_agg.orderBy(F.desc("total_accesos")).show(5)

Internet agg rows (mpio-año): 7,818


+--------+----+-------------+-------------+--------------------+--------------------+-------------+---------------------+--------------------+
|COD_MPIO| ANO|total_accesos|n_proveedores|avg_velocidad_bajada|avg_velocidad_subida|n_tecnologias|accesos_residenciales|accesos_corporativos|
+--------+----+-------------+-------------+--------------------+--------------------+-------------+---------------------+--------------------+
|   11001|2022|      9021043|           93|              332.82|              295.16|           14|                    0|              877563|
|   11001|2021|      8633348|           77|              234.96|              230.04|           14|                    0|              592435|
|   11001|2020|      8140268|           64|              153.98|              139.32|           14|                    0|              564090|
|   11001|2019|      7597590|           58|              136.52|              117.48|           14|                    0|              579285|

## 4. MEN ya está a nivel municipio-año

In [4]:
men_sel = men.select(
    "COD_MPIO", "ANO",
    "POBLACION_5_16", "TASA_MATRICULACION_5_16",
    "COBERTURA_NETA", "COBERTURA_NETA_PRIMARIA", "COBERTURA_NETA_SECUNDARIA", "COBERTURA_NETA_MEDIA",
    "DESERCION", "APROBACION", "REPROBACION", "REPITENCIA",
    "TAMANO_PROMEDIO_DE_GRUPO", "SEDES_CONECTADAS_A_INTERNET",
).filter(F.col("COD_MPIO").isNotNull())
print(f"MEN rows (mpio-año): {men_sel.count():,}")

MEN rows (mpio-año): 15,700


## 5. SISBEN es por municipio (sin año)

In [5]:
sisben_sel = sisben.select(
    "COD_MPIO",
    F.col("n_personas").alias("sisben_n_personas"),
    F.col("n_hogares").alias("sisben_n_hogares"),
    F.col("fex_total").alias("sisben_poblacion_expandida"),
    "idx_privacion",
    "pct_grupo_A", "pct_grupo_B", "pct_grupo_C", "pct_grupo_D",
    F.col("pct_rural").alias("pct_rural_sisben"),
)
print(f"SISBEN rows (mpio): {sisben_sel.count():,}")

SISBEN rows (mpio): 1,099


## 6. JOIN: panel municipal año

In [6]:
panel = (
    icfes_agg
    .join(internet_agg, on=["COD_MPIO","ANO"], how="left")
    .join(men_sel,      on=["COD_MPIO","ANO"], how="left")
    .join(broadcast(sisben_sel), on="COD_MPIO", how="left")
)
print(f"PANEL rows: {panel.count():,}")
print(f"PANEL cols: {len(panel.columns)}")
panel.printSchema()

PANEL rows: 7,004
PANEL cols: 44
root
 |-- COD_MPIO: string (nullable = true)
 |-- ANO: integer (nullable = true)
 |-- COLE_DEPTO_UBICACION: string (nullable = true)
 |-- n_estudiantes: long (nullable = false)
 |-- avg_punt_global: double (nullable = true)
 |-- sd_punt_global: double (nullable = true)
 |-- med_punt_global: integer (nullable = true)
 |-- avg_punt_lectura: double (nullable = true)
 |-- avg_punt_matematicas: double (nullable = true)
 |-- avg_punt_naturales: double (nullable = true)
 |-- avg_punt_sociales: double (nullable = true)
 |-- avg_punt_ingles: double (nullable = true)
 |-- pct_internet_icfes: double (nullable = true)
 |-- pct_computador_icfes: double (nullable = true)
 |-- pct_rural_colegio: double (nullable = true)
 |-- pct_colegio_oficial: double (nullable = true)
 |-- total_accesos: long (nullable = true)
 |-- n_proveedores: long (nullable = true)
 |-- avg_velocidad_bajada: double (nullable = true)
 |-- avg_velocidad_subida: double (nullable = true)
 |-- n_tecn

## 7. Variables derivadas: accesos per cápita, etc.

In [7]:
panel = (panel
    .withColumn("accesos_per_capita_5_16",
                F.when(F.col("POBLACION_5_16") > 0,
                       F.col("total_accesos") / F.col("POBLACION_5_16")))
    .withColumn("brecha_internet_icfes_vs_real",
                F.col("pct_internet_icfes") - F.col("accesos_per_capita_5_16"))
)

## 8. Escribir Gold Parquet (particionado por ANO)

In [8]:
import time
t0 = time.time()
(panel.write
    .mode("overwrite")
    .partitionBy("ANO")
    .option("compression","snappy")
    .parquet(P.GOLD_PANEL_MUNICIPAL))
print(f"Escrito en {time.time()-t0:.1f}s")

Escrito en 7.5s


## 9. Verificación + vista preliminar

In [9]:
g = spark.read.parquet(P.GOLD_PANEL_MUNICIPAL)
print(f"Gold rows: {g.count():,}")
print("\nTop municipios por puntaje global promedio (año más reciente):")
ano_max = g.agg(F.max("ANO")).first()[0]
print("Año más reciente:", ano_max)
g.filter(F.col("ANO") == ano_max) \
 .select("COD_MPIO","COLE_DEPTO_UBICACION","n_estudiantes","avg_punt_global",
         "pct_internet_icfes","idx_privacion","total_accesos") \
 .orderBy(F.desc("avg_punt_global")) \
 .show(15, truncate=False)

Gold rows: 7,004

Top municipios por puntaje global promedio (año más reciente):


Año más reciente: 2022


+--------+--------------------+-------------+---------------+------------------+------------------+-------------+
|COD_MPIO|COLE_DEPTO_UBICACION|n_estudiantes|avg_punt_global|pct_internet_icfes|idx_privacion     |total_accesos|
+--------+--------------------+-------------+---------------+------------------+------------------+-------------+
|25214   |CUNDINAMARCA        |1870         |305.18         |0.9209            |2.0074432452549313|42302        |
|15516   |BOYACA              |810          |295.51         |0.7531            |2.566680869194717 |30936        |
|25377   |CUNDINAMARCA        |926          |292.69         |0.9006            |2.6424427480916033|35586        |
|54518   |NORTE SANTANDER     |1044         |290.74         |0.7443            |3.7054248493097415|26909        |
|15500   |BOYACA              |54           |287.67         |0.6667            |2.750151975683891 |598          |
|15001   |BOYACA              |5141         |286.52         |0.843             |2.778321

In [10]:
spark.stop()